# Acquisition des données

## 0. Import de fonctions utiles à l'importation des données

Avant de procéder à l'acquisition des données, il est essentiel de configurer notre environnement de travail. Dans une optique de **reproductibilité** et de **clarté du code**, la logique technique (les fonctions de téléchargement) a été déportée dans un dossier dédié nommé `/fonctions`, composé de fichiers Python. 

La cellule suivante initialise cette connexion en ajoutant le dossier des scripts au chemin système de Python et en important l'ensemble des dépendances nécessaires.

In [2]:
import sys
import os

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

# On importe toutes les fonctions dans le fichier imports.py
from imports import * # type: ignore

## 1. Transfermarkt

#### **La structure des données Transfermarkt**
Ce dataset est une extraction structurée des données du site de référence mondial **Transfermarkt**, mise à disposition via le projet open-source `player-scores` de David Cariboo. Il s'agit d'une base de données combinant des faits objectifs (transferts, compositions) et des **estimations de marché** validées par un réseau d'experts. Le mode de collecte repose sur un **crowdsourcing structuré**, où les valeurs sont révisées périodiquement pour refléter l'offre et la demande réelle du football professionnel.

#### **Justification du choix**
Transfermarkt est l'autorité standard pour l'évaluation financière des joueurs. Dans ce projet, cette source est **essentielle** car elle fournit notre **variable cible** : la valeur marchande. Sans cet historique financier couplé aux données de temps de jeu, il serait impossible d'entraîner un modèle supervisé capable de comprendre les déterminants du prix d'un joueur.


#### **Utilité concrète des fichiers utilisés**
* **`player_valuations.csv` :** Contient l'historique temporel des prix, constituant ce que le modèle devra prédire.
* **`players.csv` :** Regroupe les caractéristiques intrinsèques et contractuelles : âge, taille, pied fort, et la **date d'expiration du contrat**, variable critique pour l'évaluation financière.
* **`appearances.csv` :** Recense chaque match disputé et le temps de jeu effectif (minutes), permettant de calculer le **volume d'activité** et l'importance réelle du joueur dans la rotation de son équipe.
* **`games.csv` :** Fournit les détails sur les rencontres (date, score, adversaire). Il permet de pondérer une performance individuelle selon le prestige ou la difficulté du match.
* **`clubs.csv` :** Contient des informations sur les clubs. Cela permet au modèle d'intégrer le "standing" du club employeur, car un joueur d'un "gros" club bénéficie souvent d'une surcote de marché par rapport à un joueur d'un club de milieu de tableau.
* **`competitions.csv` :** Définit le cadre de la compétition (Premier League, Ligue 1, etc.) et son niveau de prestige. C'est une variable clé pour différencier une performance réalisée dans un "Top 5" européen d'une performance dans un championnat plus faible.

#### **Hypothèses et attentes avant exploration**
1. **Relation parabolique âge/valeur :** On s'attend à un pic des valeurs marchandes entre 24 et 28 ans avant un déclin lié à la valeur de revente.
2. **Impact contractuel :** Un joueur approchant de la fin de son contrat devrait voir sa valeur marchande diminuer.
3. **Loi du temps de jeu :** Une baisse drastique des apparitions (`appearances`) sans blessure devrait signaler une perte de valeur.

In [3]:
# Configuration pour utiliser la fonction de téléchargement de données kaggle
DATASET = 'davidcariboo/player-scores'
DEST = "../data/transfermarkt_datasets"

# Appel de la fonction de téléchargement
download_kaggle_dataset(DATASET, DEST) # type: ignore

Téléchargement de davidcariboo/player-scores vers ../data/transfermarkt_datasets...
Dataset URL: https://www.kaggle.com/datasets/davidcariboo/player-scores
Téléchargement correctement effectué !


Le dataset player-scores a été téléchargé avec succès. Les fichiers principaux (joueurs et valorisations) sont désormais disponibles pour l'étape d'exploration.

## 2. FBref

#### **La structure des données FBref**
FBref fournit des statistiques de performance agrégées par saison, principalement issues des données **Opta**. Contrairement aux données transactionnelles de Transfermarkt, FBref capture l'essence technique du jeu, incluant des métriques de pointe issues de modèles probabilistes.

#### **Justification du choix**
Si Transfermarkt donne les valeurs marchandes, FBref donne les **justificatifs de performance** qui expliquent ce prix. Ces statistiques agissent comme des **variables explicatives** fondamentales. Elles permettent de distinguer un joueur efficace par chance d'un joueur créant régulièrement des occasions de haute qualité, ce qui influence directement sa valeur marchande.


#### **Utilité concrète des variables (FBref)**

Les données de FBref nous permettent de passer d'une simple observation du prix à une analyse de la **rentabilité sportive**. Voici comment chaque groupe de variables sert notre modèle de prédiction :

* **Statistiques Standard et Temps de Jeu (`MP`, `Starts`, `Min`, `Min%`, `Mn/MP`) :**
    * *Utilité :* Ces variables mesurent la **disponibilité** et le **statut** du joueur. 
    * *Impact VM :* La valeur marchande est intrinsèquement liée au temps de jeu. Un joueur avec un `Min%` élevé est un cadre dont le prix est protégé. Les colonnes comme `Mn/Sub` (minutes par entrée en jeu) permettent de distinguer les "supersubs" à fort potentiel des joueurs de complément.

* **Efficacité Offensive et Shooting (`Gls`, `Ast`, `Sh`, `SoT`, `G/SoT`) :**
    * *Utilité :* On ne se contente pas des buts. On analyse ici le volume de frappes (`Sh`) et la précision du cadrage (`SoT%`).
    * *Impact VM :* Un attaquant avec un ratio `G/Sh` (Buts par tir) élevé démontre une efficacité clinique devant le but, une caractéristique qui déclenche souvent des transferts à prix d'or.

* **Indicateurs de Performance Collective (`PPM`, `+/-`, `+/-90`, `On-Off`) :**
    * *Utilité :* Le `+/-90` mesure la différence de buts de l'équipe quand le joueur est sur le terrain par rapport à quand il est sur le banc.
    * *Impact VM :* Ces variables permettent d'identifier l'**influence réelle** d'un joueur sur les résultats de son équipe. Un joueur avec un score `On-Off` positif élevé est souvent un "élément moteur" dont la valeur dépasse ses statistiques individuelles brutes.

* **Variables de Gardien de But (`GA`, `Saves`, `Save%`, `CS%`) :**
    * *Utilité :* Pour les profils spécifiques de gardiens, nous utilisons les arrêts (`Saves`) et le pourcentage de "Clean Sheets" (`CS%`).
    * *Impact VM :* La valeur d'un gardien moderne repose sur sa capacité à maintenir un `Save%` élevé sur le long terme et à rassurer sa défense.

* **Discipline et Statistiques Diverses (`CrdY`, `CrdR`, `Fls`, `Fld`, `Int`, `TklW`) :**
    * *Utilité :* Analyse des fautes commises (`Fls`) vs subies (`Fld`), des interceptions (`Int`) et des tacles gagnés (`TklW`).
    * *Impact VM :* Le nombre de fautes subies (`Fld`) est souvent un excellent proxy pour l'agilité et la dangerosité d'un ailier (provoquant des fautes). À l'inverse, un volume de `TklW` élevé valorise les profils défensifs "propres" et efficaces.

#### **Hypothèses et attentes avant exploration**
1. **Prime à la titularisation :** Les joueurs avec un ratio `Starts/MP` proche de 1 auront une valeur marchande significativement plus stable.
2. **Corrélation Discipline/Prix :** Une accumulation de cartons rouges (`CrdR`) ou de fautes commises (`Fls`) sans volume défensif associé (`Int`) pourrait agir comme un malus sur la valeur marchande (risque disciplinaire).

In [4]:
# Configuration pour utiliser la fonction de téléchargement de données kaggle
DATASET = 'hubertsidorowicz/football-players-stats-2025-2026'
DEST = "../data/fbref_datasets"

# Appel de la fonction de téléchargement
download_kaggle_dataset(DATASET, DEST) # type: ignore

Téléchargement de hubertsidorowicz/football-players-stats-2025-2026 vers ../data/fbref_datasets...
Dataset URL: https://www.kaggle.com/datasets/hubertsidorowicz/football-players-stats-2025-2026
Téléchargement correctement effectué !


Les statistiques saisonnières 2025-2026 ont été récupérées. Ces données constituent notre socle de "performance récente". L'étape suivante consistera à effectuer une jointure avec les données de Transfermarkt en utilisant les noms de joueurs comme clé de correspondance, malgré les potentielles variations d'orthographe entre les deux sources.

## 3. Statsbomb

#### **Description de la source**
StatsBomb propose un accès "Open Data" à une partie de ses bases de données professionnelles. Contrairement aux sources précédentes, il s'agit de **données d'événements (Event Data)**. Chaque ligne représente une action technique précise (passe, tir, tacle, pression) géolocalisée sur le terrain via des coordonnées $(x, y)$.

#### **Justification du choix**
Cette source est indispensable pour capturer le **profil technique intrinsèque** du joueur. Là où FBref nous dit qu'un joueur a réussi une passe, StatsBomb nous permet de calculer la difficulté de cette passe (distance, angle, nombre d'adversaires éliminés). Cela permet d'identifier des "pépites" dont les statistiques classiques sont modestes mais dont la contribution technique à la progression du ballon est immense, justifiant ainsi des valeurs marchandes élevées ou en devenir.

#### **Utilité concrète des fichiers compilés (StatsBomb)**

* **`competitions_statsbomb.feather` :** Répertorie l'ensemble des compétitions et saisons disponibles pour assurer la cohérence temporelle et le filtrage des données.
* **`all_events.feather` :** Contient le cœur de l'analyse technique (pressions subies, longueurs de passes, localisations $x,y$) pour l'ingénierie de variables complexes.
* **`all_lineups.feather` :** Permet d'identifier les postes précis occupés par les joueurs sur le terrain et de mesurer leur polyvalence tactique.
* **`all_matches.feather` :** Fournit le contexte de chaque rencontre (adversaire, stade, date) pour lier les événements à une opposition spécifique.

#### **Hypothèses et attentes avant exploration**
1.  **Résistance au pressing :** Nous pouvons supposer que les joueurs affichant un taux de réussite élevé sous pression (`under_pressure`) possèdent une valeur marchande supérieure, car cette compétence est rare et recherchée par les clubs d'élite.
2.  **Qualité de la progression :** On s'attend à ce que les joueurs capables de réaliser des passes progressives (brisant des lignes) dans le dernier tiers du terrain voient leur valeur augmenter plus rapidement que les joueurs effectuant des passes latérales sécurisées.
3.  **Localisation et Dangerosité :** Les coordonnées des actions permettront de valider si un joueur s'approche souvent de la surface adverse, augmentant mécaniquement son attractivité financière.

Téléchargons maintenant l'ensemble des données issues de l'Open source de Statsbomb. Les fichiers sont récoltés sous le format .json pour le moment.

In [5]:
# Configuration pour utiliser la fonction de téléchargement de données git
REPO = "https://github.com/statsbomb/open-data"
DEST = "../data/statsbomb_datasets"

# Appel de la fonction de téléchargement
download_github_dataset(REPO, DEST) # type: ignore

Le dossier existe déjà. Vérification des mises à jour...
Les données sont déjà à jour !


Il s'agit maintenant de transformer ces données en fichiers .feather, plus pratique pour nous analyses futures.

In [11]:
# Dossier où on stocke tes fichiers finaux
destination = "../data/statsbomb_datasets/data"

In [ ]:
# Les compétitions

compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data", 
    output_folder_path=destination,
    output_name="competitions_statsbomb",
    recursive=False
)

Traitement de 1 fichiers trouvés dans data...
Fusion et sauvegarde...
Terminé ! Fichier : competitions_statsbomb.feather (75 lignes)


In [ ]:
# Les lineups

compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data/lineups", 
    output_folder_path=destination,
    output_name="all_lineups",
    record_path=['lineup'],
    meta=['team_name', 'team_id'],
    recursive = False
)

Traitement de 3464 fichiers trouvés dans lineups...
Fusion et sauvegarde...
Terminé ! Fichier : all_lineups.feather (131901 lignes)


In [ ]:
# Les events

# On ne garde que les colonnes essentielles car il y a trop d'informations dans ces fichiers.
cols_events = [
    'match_id', 'id', 'index', 'period', 'timestamp', 'minute', 'second', 
    'type.name', 'team.name', 'player.name', 'position.name', 
    'location', 'duration', 'under_pressure', 'pass.end_location', 
    'pass.outcome.name', 'shot.statsbomb_xg', 'shot.outcome.name'
]

compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data/events", 
    output_folder_path=destination,
    output_name="all_events",
    columns_to_keep=cols_events,
    recursive = False
)

Traitement de 3464 fichiers trouvés dans events...
Fusion et sauvegarde...
Terminé ! Fichier : all_events.feather (12188949 lignes)


In [ ]:
# Les matches

# Les colonnes essentielles pour les matches
cols_matches = [
    'match_id', 'match_date', 'kick_off', 'competition.competition_name', 
    'season.season_name', 'home_team.home_team_name', 'away_team.away_team_name', 
    'home_score', 'away_score'
]


compile_statsbomb_to_feather(
    json_folder_path="../data/statsbomb_datasets/data/matches", 
    output_folder_path=destination,
    output_name="all_matches",
    columns_to_keep=cols_matches,
    recursive = True
)

Traitement de 75 fichiers trouvés dans matches...
Fusion et sauvegarde...
Terminé ! Fichier : all_matches.feather (3464 lignes)


Les fichiers JSON volumineux ont été transformés en format .feather pour optimiser la vitesse de lecture et l'usage de la mémoire RAM.

## 4. football-data.co.uk

#### **Description de la source**
Ce site collecte les résultats détaillés des matchs de football (scores, statistiques de match) ainsi que les **cotes de paris sportifs** provenant des principaux bookmakers mondiaux. Il s'agit d'une source de nature "résultats et probabilités".

#### **Justification du choix**
La valeur d'un joueur n'est pas isolée ; elle est fortement influencée par la **force de son équipe** et la difficulté de son championnat. Les cotes de paris sportifs servent ici de "proxy" (indicateur indirect) pour mesurer la dominance d'un club et le prestige d'une rencontre. Intégrer ces données permet de pondérer les performances individuelles par le niveau collectif de l'équipe du joueur.

#### **Utilité concrète des variables (football-data.co.uk)**

L'intégration des données de `football-data.co.uk` permet d'ajouter une dimension contextuelle et macro-économique indispensable. Au-delà des performances individuelles, la valeur marchande d'un joueur est étroitement liée à la **santé compétitive** de son club et à la perception du marché par les bookmakers.

* **Dynamique de Score et Résultats (`FTHG`, `FTAG`, `FTR`, `HTHG`, `HTAG`, `HTR`) :**
    * *Rôle :* Scores finaux et à la mi-temps, ainsi que le résultat final (Home/Draw/Away).
    * *Impact VM :* Ces variables permettent de calculer des séries de victoires ou de défaites. Un joueur évoluant dans une équipe qui "gagne" (notamment les matchs à enjeux révélés par les scores à la mi-temps) voit sa valeur marchande protégée par la dynamique collective positive.

* **Statistiques de Domination Collective (`HS`, `AS`, `HST`, `AST`, `HC`, `AC`) :**
    * *Rôle :* Tirs (`Shots`), tirs cadrés (`Shots on Target`) et corners obtenus par chaque équipe.
    * *Impact VM :* Ces données permettent de définir le **style de jeu de l'équipe**. Un joueur évoluant dans une équipe qui génère un haut volume de corners (`HC/AC`) et de tirs cadrés possède plus d'opportunités de briller statistiquement, ce qui influence indirectement son exposition médiatique et sa valeur marchande.

* **Discipline et Arbitrage (`HF`, `AF`, `HY`, `AY`, `HR`, `AR`, `Referee`) :**
    * *Rôle :* Fautes commises, cartons jaunes/rouges et identité de l'arbitre.
    * *Impact VM :* Permet d'évaluer l'agressivité collective de l'équipe. L'identité de l'arbitre (`Referee`) peut servir à normaliser les statistiques disciplinaires des joueurs (certains arbitres étant plus sévères, cela permet de ne pas pénaliser injustement la valeur d'un joueur).

* **Analyse des Cotes et Probabilités Marché (`B365H`, `BWH`, `IWH`, `PSH`, `WHH`, `VCH`, etc.) :**
    * *Rôle :* Cotes d'ouverture et de fermeture (suffixes `CH`, `CD`, `CA`) provenant de multiples bookmakers (Bet365, BW, Pinnacle, William Hill).
    * *Impact VM :* Les cotes faibles (`AvgH` faible pour le club du joueur) indiquent que le joueur évolue dans une équipe **dominante** (favorite). Le marché des transferts applique souvent une "prime de standing" aux joueurs des clubs favoris.

* **Indicateurs de Flux et Over/Under (`B365>2.5`, `AvgC<2.5`, `AHh`, `AvgCAHA`) :**
    * *Rôle :* Cotes sur le nombre de buts.
    * *Impact VM :* Un joueur évoluant dans une équipe impliquée dans des matchs à "Over 2.5 buts" (spectacle offensif) sera plus exposé aux recruteurs.

#### **Hypothèses et attentes avant exploration**
1. **Prime de Dominance :** À performance égale, un joueur évoluant dans une équipe dont la cote moyenne est inférieure à 1.50 aura une valeur marchande supérieure par rapport à un joueur d'une équipe "outsider".
2. **Impact de la "Forme" Collective :** Une corrélation positive est attendue entre le ratio de victoires (`FTR`) et la hausse de la valeur marchande lors de la mise à jour suivante sur Transfermarkt.
3. **Poids du Championnat (`Div`, `league`) :** Nous supposerons l'hypothèse d'une inflation structurelle des prix pour les joueurs de Premier League (`E0`) par rapport aux autres ligues du Big 5, même pour des équipes de bas de tableau.

In [12]:
# On sélectionne les données qui nous intéressent
# Ici, on choisit les données issues du Big 5 lors des 3 dernières saisons
seasons = ["2324","2425", "2526"]
leagues = ["F1", "E0", "SP1", "I1", "D1"]

# Appel de la fonction de téléchargement
download_football_data_datasets(seasons, leagues) # type: ignore

c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\imports.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["league"] = league
c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\imports.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["league"] = league
c:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\predict_vm_football\fonctions\imports.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calli

Téléchargement terminé !


Les données de résultats et de cotes pour les 3 dernières saisons du Big 5 sont prêtes. Nous disposons désormais du contexte collectif nécessaire pour pondérer les performances individuelles des joueurs par la force de leur équipe respective.